# 02 — Model Comparison
Carica i dati preprocessati e confronta: SVM, XGBoost, MLP, FusionWithCrossAttention.


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

OUTPUT_PATH = 'processed_data'
N_SPLITS    = 5
RANDOM_STATE = 42

In [3]:
# ── CARICA DATI PREPROCESSATI ────────────────────────────────────────
X_visual  = np.load(f'{OUTPUT_PATH}/X_visual_pca.npy')   # (N, 64)
X_audio   = np.load(f'{OUTPUT_PATH}/X_audio.npy')        # (N, 128)
y_encoded = np.load(f'{OUTPUT_PATH}/y_encoded.npy')      # (N,)
classes   = np.load(f'{OUTPUT_PATH}/label_classes.npy', allow_pickle=True)

# Feature concatenate per i modelli ML classici
X_combined = np.hstack([X_visual, X_audio])  # (N, 192)

print(f'X_visual  : {X_visual.shape}')
print(f'X_audio   : {X_audio.shape}')
print(f'X_combined: {X_combined.shape}')
print(f'Classi    : {classes}')

# Class weights globali (usati da tutti i modelli)
class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f'Class weights: {class_weight_dict}')

X_visual  : (446, 64)
X_audio   : (446, 128)
X_combined: (446, 192)
Classi    : ['Negative' 'Neutral' 'Positive']
Class weights: {0: np.float64(2.858974358974359), 1: np.float64(1.0931372549019607), 2: np.float64(0.5762273901808785)}


In [4]:
# ── UTILITY: cross-validation per modelli sklearn ────────────────────
def evaluate_sklearn_model(model_fn, X, y, n_splits=N_SPLITS, use_smote=True):
    """Valuta un modello sklearn con StratifiedKFold + SMOTE opzionale."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_f1, fold_acc = [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_tr  = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        if use_smote:
            sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        model = model_fn()
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        fold_f1.append(f1_score(y_val, preds, average='macro', zero_division=0))
        fold_acc.append(accuracy_score(y_val, preds))

    return np.mean(fold_f1), np.mean(fold_acc), np.std(fold_f1)

In [5]:
# ── 1. SVM ───────────────────────────────────────────────────────────
svm_f1, svm_acc, svm_std = evaluate_sklearn_model(
    lambda: SVC(kernel='rbf', class_weight='balanced', C=10, gamma='scale', random_state=RANDOM_STATE),
    X_combined, y_encoded
)
print(f'SVM          F1-macro: {svm_f1:.4f} ± {svm_std:.4f}  |  ACC: {svm_acc:.4f}')

SVM          F1-macro: 0.4299 ± 0.0173  |  ACC: 0.5919


In [6]:
# ── 2. RANDOM FOREST ─────────────────────────────────────────────────
rf_f1, rf_acc, rf_std = evaluate_sklearn_model(
    lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                   random_state=RANDOM_STATE, n_jobs=-1),
    X_combined, y_encoded
)
print(f'RandomForest F1-macro: {rf_f1:.4f} ± {rf_std:.4f}  |  ACC: {rf_acc:.4f}')

RandomForest F1-macro: 0.4044 ± 0.0326  |  ACC: 0.5762


In [7]:
# ── 3. XGBOOST ───────────────────────────────────────────────────────
# scale_pos_weight gestisce lo sbilanciamento nativamente per XGBoost
neg_count = np.sum(y_encoded == 0)
pos_count = np.sum(y_encoded == 2)

xgb_f1, xgb_acc, xgb_std = evaluate_sklearn_model(
    lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          use_label_encoder=False, eval_metric='mlogloss',
                          random_state=RANDOM_STATE, n_jobs=-1),
    X_combined, y_encoded
)
print(f'XGBoost      F1-macro: {xgb_f1:.4f} ± {xgb_std:.4f}  |  ACC: {xgb_acc:.4f}')

XGBoost      F1-macro: 0.4253 ± 0.0521  |  ACC: 0.5718


In [8]:
# ── ARCHITETTURE NEURALI ─────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=3.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class SimpleMLP(nn.Module):
    """Baseline neurale: nessuna fusione elaborata, solo concatenazione."""
    def __init__(self, visual_dim, audio_dim, hidden=128, num_classes=3, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(visual_dim + audio_dim, hidden),
            nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, v, a):
        return self.net(torch.cat([v, a], dim=1))


class CrossModalAttention(nn.Module):
    def __init__(self, dim=128, num_heads=2, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                              dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q_feats, kv_feats):
        q = q_feats.unsqueeze(1)
        k = kv_feats.unsqueeze(1)
        attended, _ = self.attn(q, k, k)
        return self.norm(q_feats + self.dropout(attended.squeeze(1)))


class FusionWithCrossAttention(nn.Module):
    def __init__(self, visual_dim, audio_dim, shared_dim=128, num_classes=3,
                 num_heads=2, dropout=0.4):
        super().__init__()
        self.visual_enc = nn.Sequential(
            nn.Linear(visual_dim, shared_dim),
            nn.LayerNorm(shared_dim), nn.GELU(), nn.Dropout(dropout)
        )
        self.audio_enc = nn.Sequential(
            nn.Linear(audio_dim, shared_dim),
            nn.LayerNorm(shared_dim), nn.GELU(), nn.Dropout(dropout)
        )
        self.cross_v = CrossModalAttention(shared_dim, num_heads, dropout)
        self.cross_a = CrossModalAttention(shared_dim, num_heads, dropout)
        self.gate    = nn.Linear(shared_dim * 2, shared_dim)
        self.classifier = nn.Sequential(
            nn.Linear(shared_dim, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, v, a):
        v = self.visual_enc(v)
        a = self.audio_enc(a)
        v_att = self.cross_v(v, a)
        a_att = self.cross_a(a, v)
        gate  = torch.sigmoid(self.gate(torch.cat([v_att, a_att], dim=1)))
        return self.classifier(gate * v_att + (1 - gate) * a_att)

In [9]:
# ── UTILITY: training loop neurale ───────────────────────────────────
def train_neural_model(model_fn, X_v, X_a, y,
                        max_epochs=200, patience=20, lr=0.0001,
                        batch_size=32, n_splits=N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    cw  = torch.tensor(class_weights, dtype=torch.float32)
    fold_f1, fold_acc = [], []

    for train_idx, val_idx in skf.split(X_v, y):
        Xv_tr, Xv_val = X_v[train_idx], X_v[val_idx]
        Xa_tr, Xa_val = X_a[train_idx], X_a[val_idx]
        y_tr,  y_val  = y[train_idx],   y[val_idx]

        # SMOTE sulla concatenazione
        X_comb = np.hstack([Xv_tr, Xa_tr])
        sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_tr = sm.fit_resample(X_comb, y_tr)
        v_dim = Xv_tr.shape[1]
        Xv_tr = X_res[:, :v_dim]
        Xa_tr = X_res[:, v_dim:]

        sc_v = StandardScaler(); Xv_tr = sc_v.fit_transform(Xv_tr); Xv_val = sc_v.transform(Xv_val)
        sc_a = StandardScaler(); Xa_tr = sc_a.fit_transform(Xa_tr); Xa_val = sc_a.transform(Xa_val)

        def to_loader(Xv, Xa, yt, shuffle):
            return DataLoader(TensorDataset(
                torch.tensor(Xv, dtype=torch.float32),
                torch.tensor(Xa, dtype=torch.float32),
                torch.tensor(yt, dtype=torch.long)
            ), batch_size=batch_size, shuffle=shuffle)

        train_loader = to_loader(Xv_tr, Xa_tr, y_tr, True)
        val_loader   = to_loader(Xv_val, Xa_val, y_val, False)

        model     = model_fn(Xv_tr.shape[1], Xa_tr.shape[1])
        criterion = FocalLoss(weight=cw, gamma=3.0)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-6)

        best_f1, no_improve, best_state = 0.0, 0, None

        for epoch in range(max_epochs):
            model.train()
            for iv, ia, lbl in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(iv, ia), lbl)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            vp, vt = [], []
            with torch.no_grad():
                for iv, ia, lbl in val_loader:
                    _, p = torch.max(model(iv, ia), 1)
                    vp.extend(p.numpy()); vt.extend(lbl.numpy())

            vf1 = f1_score(vt, vp, average='macro', zero_division=0)
            scheduler.step(vf1)

            if vf1 > best_f1:
                best_f1, no_improve, best_state = vf1, 0, model.state_dict()
            else:
                no_improve += 1
            if no_improve >= patience:
                break

        model.load_state_dict(best_state)
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for iv, ia, lbl in val_loader:
                _, p = torch.max(model(iv, ia), 1)
                preds.extend(p.numpy()); trues.extend(lbl.numpy())

        fold_f1.append(f1_score(trues, preds, average='macro', zero_division=0))
        fold_acc.append(accuracy_score(trues, preds))

    return np.mean(fold_f1), np.mean(fold_acc), np.std(fold_f1)

In [10]:
# ── 4. MLP SEMPLICE (baseline neurale) ───────────────────────────────
mlp_f1, mlp_acc, mlp_std = train_neural_model(
    lambda vd, ad: SimpleMLP(vd, ad, hidden=128, dropout=0.4),
    X_visual, X_audio, y_encoded
)
print(f'MLP          F1-macro: {mlp_f1:.4f} ± {mlp_std:.4f}  |  ACC: {mlp_acc:.4f}')

MLP          F1-macro: 0.1752 ± 0.0488  |  ACC: 0.2039


In [11]:
# ── 5. FUSION WITH CROSS ATTENTION ───────────────────────────────────
fusion_f1, fusion_acc, fusion_std = train_neural_model(
    lambda vd, ad: FusionWithCrossAttention(vd, ad, shared_dim=128, num_heads=2, dropout=0.4),
    X_visual, X_audio, y_encoded
)
print(f'FusionCrossA F1-macro: {fusion_f1:.4f} ± {fusion_std:.4f}  |  ACC: {fusion_acc:.4f}')

FusionCrossA F1-macro: 0.2429 ± 0.0414  |  ACC: 0.2734


In [12]:
# ── RIEPILOGO FINALE ──────────────────────────────────────────────────
summary = pd.DataFrame([
    {'Modello': 'SVM (RBF)',              'F1-macro': svm_f1,    'Std': svm_std,    'Accuracy': svm_acc},
    {'Modello': 'Random Forest',          'F1-macro': rf_f1,     'Std': rf_std,     'Accuracy': rf_acc},
    {'Modello': 'XGBoost',               'F1-macro': xgb_f1,    'Std': xgb_std,    'Accuracy': xgb_acc},
    {'Modello': 'MLP (baseline)',         'F1-macro': mlp_f1,    'Std': mlp_std,    'Accuracy': mlp_acc},
    {'Modello': 'FusionCrossAttention',   'F1-macro': fusion_f1, 'Std': fusion_std, 'Accuracy': fusion_acc},
]).sort_values('F1-macro', ascending=False)

print('\n═══════════ RIEPILOGO ═══════════')
print(summary.to_string(index=False, float_format='{:.4f}'.format))
print(f"\n🏆 Miglior modello: {summary.iloc[0]['Modello']}  (F1-macro: {summary.iloc[0]['F1-macro']:.4f})")


═══════════ RIEPILOGO ═══════════
             Modello  F1-macro    Std  Accuracy
           SVM (RBF)    0.4299 0.0173    0.5919
             XGBoost    0.4253 0.0521    0.5718
       Random Forest    0.4044 0.0326    0.5762
FusionCrossAttention    0.2429 0.0414    0.2734
      MLP (baseline)    0.1752 0.0488    0.2039

🏆 Miglior modello: SVM (RBF)  (F1-macro: 0.4299)
